# Prepare Dataset: Sequences → PyG Graphs → Train / Val / Test Splits

This notebook takes the outputs from `Sequencer.ipynb` and `Logs2Graphs.ipynb` and produces
ready-to-train datasets for:
- **Graph-first GNN** (PyTorch Geometric `Data` objects with node + edge features)
- **Sequence-first model** (padded cluster-id sequences with embeddings)

**Pipeline:**
1. Load sequences + enriched templates + labels
2. Compute hybrid embeddings (TF-IDF + SBERT)
3. Build collapsed graphs with positional features → convert to PyG `Data`
4. Build padded embedding sequences for the sequence model
5. Stratified split into train / val / test (70 / 15 / 15)
6. Save everything to `data/processed/`

In [ ]:
import os, sys, json, hashlib, time, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

import torch
from torch_geometric.data import Data
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath(".."))

PROCESSED_DIR = Path("../data/processed/BGL")
RAW_DIR       = Path("../data/raw")
SEED          = 42

print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1 — Load sequences, enriched templates, and labels

In [ ]:
# ── Auto-pick newest sequence file ────────────────────────────────────────────
seq_candidates = sorted(PROCESSED_DIR.glob("*_bgl_sequences.parquet"))
if not seq_candidates:
    raise FileNotFoundError("No *_bgl_sequences.parquet — run Sequencer.ipynb first.")

seq_file = seq_candidates[-1]
RUN_TAG  = seq_file.stem.replace("_bgl_sequences", "")

df_blocks = pd.read_parquet(seq_file, engine="fastparquet")
df_blocks["parameters"] = df_blocks["parameters"].apply(json.loads)
sequences = {bid: grp for bid, grp in df_blocks.groupby("window_id", sort=False)}

print(f"Run tag          : {RUN_TAG}")
print(f"Sequence file    : {seq_file}")
print(f"Total rows       : {len(df_blocks):,}")
print(f"Unique blocks    : {len(sequences):,}")

In [ ]:
# ── Load enriched templates ───────────────────────────────────────────────────
tmpl_candidates = sorted(PROCESSED_DIR.glob("*_bgl_templates_enriched.json"))
if not tmpl_candidates:
    tmpl_candidates = sorted(PROCESSED_DIR.glob("*_bgl_templates.json"))

tmpl_file = tmpl_candidates[-1]
with open(tmpl_file) as f:
    templates_data = json.load(f)

all_cids      = [t["cluster_id"] for t in templates_data]
all_templates  = [t["template"] for t in templates_data]
cluster_to_template = {t["cluster_id"]: t["template"] for t in templates_data}

cluster_to_enriched = {}
for t in templates_data:
    if "enriched_large" in t:
        try:
            cluster_to_enriched[t["cluster_id"]] = json.loads(t["enriched_large"])
        except (json.JSONDecodeError, TypeError):
            pass

print(f"Templates loaded : {len(templates_data)}")
print(f"With enrichment  : {len(cluster_to_enriched)}")

In [ ]:
# ── Load anomaly labels ───────────────────────────────────────────────────────
# BGL labels are inline: the first token of each log line is '-' for normal
# or an alert code (e.g. 'KERNDTLB', 'APPSEV') for anomalous.
# BGLParser already captures this as the boolean `is_anomaly` column.
# A window is anomalous if ANY of its lines is anomalous (from Sequencer).

window_label = df_blocks.groupby("window_id")["is_anomaly"].any()
block_labels = window_label.astype(int).to_dict()  # window_id (int) → 0 or 1

LABELS_AVAILABLE = True  # ground-truth labels, not heuristic

n_total = len(block_labels)
n_anom  = sum(block_labels.values())
n_norm  = n_total - n_anom

print(f"Labels derived from inline BGL alert codes (is_anomaly column)")
print(f"  Total windows  : {n_total:,}")
print(f"  Normal         : {n_norm:,}  ({100 * n_norm / n_total:.2f}%)")
print(f"  Anomalous      : {n_anom:,}  ({100 * n_anom / n_total:.2f}%)")

## 2 — Compute hybrid embeddings (TF-IDF + SBERT)

In [ ]:
# ── 2a. TF-IDF on raw templates ──────────────────────────────────────────────
tfidf_vec = TfidfVectorizer(analyzer="word", token_pattern=r"[^\s]+")
tfidf_matrix = tfidf_vec.fit_transform(all_templates)
tfidf_dense = tfidf_matrix.toarray()

print(f"TF-IDF: {tfidf_dense.shape[1]}-dim vectors for {tfidf_dense.shape[0]} templates")

# ── 2b. Sentence-BERT on enriched text ───────────────────────────────────────
try:
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

    enriched_texts = []
    for cid in all_cids:
        info = cluster_to_enriched.get(cid)
        if info:
            parts = [info.get("purpose", "")]
            for fm in info.get("failure_modes", []):
                parts.append(f"{fm.get('name', '')}: {fm.get('description', '')}")
            enriched_texts.append(" ".join(parts))
        else:
            enriched_texts.append(cluster_to_template.get(cid, ""))

    sbert_embeddings = sbert_model.encode(enriched_texts, show_progress_bar=True, normalize_embeddings=True)
    hybrid_embeddings = np.hstack([tfidf_dense, sbert_embeddings])
    print(f"Hybrid: {hybrid_embeddings.shape}  (TF-IDF {tfidf_dense.shape[1]}d + SBERT {sbert_embeddings.shape[1]}d)")
except ImportError:
    print("sentence-transformers not available — using TF-IDF only")
    hybrid_embeddings = tfidf_dense

# Map: cluster_id → embedding vector
cluster_embeddings = {
    cid: hybrid_embeddings[i]
    for i, cid in enumerate(all_cids)
}
EMBED_DIM = hybrid_embeddings.shape[1]
print(f"\nEmbedding dim: {EMBED_DIM}")
print(f"Lookup ready for {len(cluster_embeddings)} templates")

## 3 — Build collapsed graphs → PyG `Data` objects

Each block becomes one `torch_geometric.data.Data` with:
- `x`: node feature matrix `(num_nodes, node_dim)` — embedding + positional + count features
- `edge_index`: `(2, num_edges)` COO adjacency
- `edge_attr`: `(num_edges, edge_dim)` — time-delta distribution + positional features
- `y`: anomaly label (0 or 1)
- `block_id`: for traceability

In [ ]:
def _is_numeric(x) -> bool:
    """Return True only for finite numeric strings.

    float("nan") and float("inf") do NOT raise ValueError, so a bare
    try/except is insufficient — we must explicitly reject non-finite values.
    """
    try:
        v = float(x)
        return np.isfinite(v)   # rejects nan, +inf, -inf
    except (ValueError, TypeError):
        return False


# ── Node feature order ────────────────────────────────────────────────────────
# [embedding (EMBED_DIM)] + [occurrence_count, param_count, param_num_mean, param_num_max,
#  first_pos, last_pos, mean_pos, std_pos, pos_spread]  = EMBED_DIM + 9
NODE_EXTRA_DIM = 9
NODE_DIM = EMBED_DIM + NODE_EXTRA_DIM

# ── Edge feature order ────────────────────────────────────────────────────────
# [weight, td_min, td_p25, td_median, td_p75, td_max, td_std,
#  mean_src_pos, mean_dst_pos, mean_pos_delta]  = 10
EDGE_DIM = 10

print(f"Node feature dim : {NODE_DIM}  (embedding {EMBED_DIM} + {NODE_EXTRA_DIM} numeric)")
print(f"Edge feature dim : {EDGE_DIM}")

In [ ]:
def seq_to_pyg(seq: pd.DataFrame, label: int) -> Data:
    """Convert a single window's sequence DataFrame to a PyG Data object."""
    wid    = seq["window_id"].iloc[0]
    cids   = seq["cluster_id"].tolist()
    params = seq["parameters"].tolist()
    ts     = seq["unix_ts"].tolist()
    n      = len(cids)

    # ── Aggregate per-node ────────────────────────────────────────────────────
    node_params    = defaultdict(list)
    node_count     = Counter(cids)
    node_positions = defaultdict(list)

    for i, (cid, p) in enumerate(zip(cids, params)):
        node_params[cid].extend(p)
        node_positions[cid].append(i / max(n - 1, 1))

    unique_cids = list(dict.fromkeys(cids))
    cid_to_idx  = {cid: idx for idx, cid in enumerate(unique_cids)}
    num_nodes   = len(unique_cids)

    node_feats = np.zeros((num_nodes, NODE_DIM), dtype=np.float32)
    for idx, cid in enumerate(unique_cids):
        emb  = cluster_embeddings.get(cid, np.zeros(EMBED_DIM))
        # _is_numeric now rejects nan/inf strings, so nums contains only finite values
        nums = [float(x) for x in node_params[cid] if _is_numeric(x)]
        pos  = np.array(node_positions[cid])

        node_feats[idx, :EMBED_DIM] = emb
        node_feats[idx, EMBED_DIM + 0] = node_count[cid]
        node_feats[idx, EMBED_DIM + 1] = len(node_params[cid])
        node_feats[idx, EMBED_DIM + 2] = float(np.mean(nums)) if nums else 0.0
        node_feats[idx, EMBED_DIM + 3] = float(np.max(nums))  if nums else 0.0
        node_feats[idx, EMBED_DIM + 4] = float(pos.min())
        node_feats[idx, EMBED_DIM + 5] = float(pos.max())
        node_feats[idx, EMBED_DIM + 6] = float(pos.mean())
        node_feats[idx, EMBED_DIM + 7] = float(pos.std()) if len(pos) > 1 else 0.0
        node_feats[idx, EMBED_DIM + 8] = float(pos.max() - pos.min())

    # Safety: replace any residual NaN/Inf (e.g. from degenerate embeddings)
    node_feats = np.nan_to_num(node_feats, nan=0.0, posinf=0.0, neginf=0.0)

    # ── Aggregate per-edge ────────────────────────────────────────────────────
    edge_deltas  = defaultdict(list)
    edge_src_pos = defaultdict(list)
    edge_dst_pos = defaultdict(list)

    for i in range(n - 1):
        src, dst = cids[i], cids[i + 1]
        src_norm = i / max(n - 1, 1)
        dst_norm = (i + 1) / max(n - 1, 1)
        edge_src_pos[(src, dst)].append(src_norm)
        edge_dst_pos[(src, dst)].append(dst_norm)

        if ts[i] is not None and ts[i + 1] is not None:
            edge_deltas[(src, dst)].append(float(ts[i + 1] - ts[i]))
        elif (src, dst) not in edge_deltas:
            edge_deltas[(src, dst)]

    src_list, dst_list, edge_feats_list = [], [], []
    for (src, dst) in edge_src_pos:
        deltas = edge_deltas.get((src, dst), [])
        s_pos  = np.array(edge_src_pos[(src, dst)])
        d_pos  = np.array(edge_dst_pos[(src, dst)])

        ef = np.zeros(EDGE_DIM, dtype=np.float32)
        ef[0] = len(s_pos)
        if deltas:
            arr = np.array(deltas)
            ef[1] = arr.min()
            ef[2] = np.percentile(arr, 25)
            ef[3] = np.median(arr)
            ef[4] = np.percentile(arr, 75)
            ef[5] = arr.max()
            ef[6] = arr.std()
        else:
            ef[1:7] = [-1, -1, -1, -1, -1, 0]
        ef[7] = s_pos.mean()
        ef[8] = d_pos.mean()
        ef[9] = (d_pos - s_pos).mean()

        src_list.append(cid_to_idx[src])
        dst_list.append(cid_to_idx[dst])
        edge_feats_list.append(ef)

    x          = torch.from_numpy(node_feats)
    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_attr  = torch.from_numpy(np.array(edge_feats_list)) if edge_feats_list else torch.zeros((0, EDGE_DIM))
    y          = torch.tensor([label], dtype=torch.long)

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        window_id=wid,
        num_nodes=num_nodes,
    )


In [ ]:
# ── Build all PyG Data objects ────────────────────────────────────────────────
from tqdm.auto import tqdm

t0 = time.time()
all_data = []
skipped  = 0

for wid, seq in tqdm(sequences.items(), desc="Building PyG graphs"):
    label = block_labels.get(wid, 0)
    try:
        data = seq_to_pyg(seq, label)
        all_data.append(data)
    except Exception as e:
        skipped += 1
        if skipped <= 3:
            print(f"  ⚠ Skipped window {wid}: {e}")

elapsed = time.time() - t0
print(f"\nBuilt {len(all_data):,} PyG graphs in {elapsed:.1f}s  (skipped {skipped})")

# Quick sanity check
sample = all_data[0]
print(f"\nSample graph: {sample}")
print(f"  x shape        : {sample.x.shape}")
print(f"  edge_index     : {sample.edge_index.shape}")
print(f"  edge_attr      : {sample.edge_attr.shape}")
print(f"  y              : {sample.y}")
print(f"  window_id      : {sample.window_id}")


## 4 — Build padded cluster-id sequences (memory-efficient)

Instead of materialising the full `(N, MAX_SEQ_LEN, EMBED_DIM)` float32 tensor
(which can easily exceed available RAM), we store **integer cluster-id sequences**
`(N, MAX_SEQ_LEN)` as int16 — only ~2 bytes per position.

Dense embedding matrices are reconstructed on-the-fly during training via a
simple lookup table, or written to disk in chunks during saving.

In [ ]:
# ── Compute sequence stats to choose padding length ──────────────────────────
seq_lens = np.array([len(seq) for seq in sequences.values()])

p95 = int(np.percentile(seq_lens, 95))
p99 = int(np.percentile(seq_lens, 99))
MAX_SEQ_LEN = p99  # truncate anything longer; pad anything shorter

print(f"Sequence length stats:")
print(f"  min={seq_lens.min()}, median={int(np.median(seq_lens))}, "
      f"mean={seq_lens.mean():.1f}, p95={p95}, p99={p99}, max={seq_lens.max()}")
print(f"  MAX_SEQ_LEN = {MAX_SEQ_LEN} (p99 — {100 * (seq_lens <= MAX_SEQ_LEN).mean():.1f}% of blocks fit)")

In [ ]:
# ── Build compact padded sequences (int16 cluster-id matrices) ───────────────
# Stores only cluster ids — dense embeddings are reconstructed on-the-fly.
# Memory: N × MAX_SEQ_LEN × 2 bytes  (vs. N × MAX_SEQ_LEN × EMBED_DIM × 4)

PAD_ID = -1  # padding value (no valid cluster_id is negative)

t0 = time.time()
seq_block_ids = []
seq_lengths   = []
seq_labels    = []
seq_cids      = np.full((len(sequences), MAX_SEQ_LEN), PAD_ID, dtype=np.int16)

for row_idx, (bid, seq) in enumerate(
    tqdm(sequences.items(), desc="Building padded sequences")
):
    cids = seq["cluster_id"].tolist()
    actual_len = min(len(cids), MAX_SEQ_LEN)
    seq_cids[row_idx, :actual_len] = cids[:actual_len]

    seq_block_ids.append(bid)
    seq_lengths.append(actual_len)
    seq_labels.append(block_labels.get(bid, 0))

seq_lengths = np.array(seq_lengths, dtype=np.int32)
seq_y       = np.array(seq_labels, dtype=np.int32)

elapsed = time.time() - t0
print(f"\nBuilt {len(seq_cids):,} compact padded sequences in {elapsed:.1f}s")
print(f"  Shape: {seq_cids.shape}  ({seq_cids.nbytes / 1e6:.1f} MB — "
      f"vs ~{len(seq_cids) * MAX_SEQ_LEN * EMBED_DIM * 4 / 1e9:.1f} GB dense)")
print(f"  Label distribution: normal={np.sum(seq_y == 0):,}, anomalous={np.sum(seq_y == 1):,}")

## 5 — Stratified train / val / test split (70 / 15 / 15)

The HDFS dataset is heavily imbalanced (~2.9% anomalous).  
Stratification ensures each split has the same anomaly ratio.

In [ ]:
# ── Collect labels for all data objects, aligned by index ─────────────────────
all_window_ids = [d.window_id for d in all_data]
all_labels     = np.array([d.y.item() for d in all_data])
indices        = np.arange(len(all_data))

# ── First split: train (70%) vs temp (30%) ────────────────────────────────────
idx_train, idx_temp = train_test_split(
    indices, test_size=0.30, random_state=SEED, stratify=all_labels
)

# ── Second split: val (15%) vs test (15%) from temp ───────────────────────────
labels_temp = all_labels[idx_temp]
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=SEED, stratify=labels_temp
)

print(f"Split sizes (total {len(all_data):,}):")
for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    labs = all_labels[idx]
    n_anom = labs.sum()
    print(f"  {name:>5}: {len(idx):>7,}  (anomalous: {n_anom:>5,} = {100*n_anom/len(idx):.2f}%)")


In [ ]:
# ── Build split datasets ─────────────────────────────────────────────────────

# Graph datasets (for GNN)
graph_train = [all_data[i] for i in idx_train]
graph_val   = [all_data[i] for i in idx_val]
graph_test  = [all_data[i] for i in idx_test]

# Sequence datasets (for LSTM / sequence model)
# Map window_id → row index in the compact sequence arrays
wid_to_seq_idx = {wid: i for i, wid in enumerate(seq_block_ids)}

def gather_seq_split(graph_indices):
    seq_idxs = [wid_to_seq_idx[all_window_ids[gi]] for gi in graph_indices]
    return seq_cids[seq_idxs], seq_lengths[seq_idxs], seq_y[seq_idxs]

seq_train_cids, seq_train_len, seq_train_y = gather_seq_split(idx_train)
seq_val_cids,   seq_val_len,   seq_val_y   = gather_seq_split(idx_val)
seq_test_cids,  seq_test_len,  seq_test_y  = gather_seq_split(idx_test)

print("Graph splits:")
print(f"  train: {len(graph_train):,}  val: {len(graph_val):,}  test: {len(graph_test):,}")
print(f"\nSequence splits (compact int16 cluster-ids):")
print(f"  train: {seq_train_cids.shape}  val: {seq_val_cids.shape}  test: {seq_test_cids.shape}")


In [ ]:
# ── Visualise split label distributions ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, labs) in zip(axes, [
    ("Train", all_labels[idx_train]),
    ("Val",   all_labels[idx_val]),
    ("Test",  all_labels[idx_test]),
]):
    counts = [np.sum(labs == 0), np.sum(labs == 1)]
    bars = ax.bar(["Normal", "Anomalous"], counts, color=["steelblue", "tomato"])
    ax.set_title(f"{name}  (n={len(labs):,})")
    ax.set_ylabel("Count")
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(labs)*0.005,
                f"{c:,}\n({100*c/len(labs):.1f}%)", ha="center", va="bottom", fontsize=9)

fig.suptitle(f"Label distribution across splits  (seed={SEED})", fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Save everything to `data/processed/`

Outputs:
- `{RUN_TAG}_graph_dataset.pt` — list of PyG `Data` + split indices
- `{RUN_TAG}_seq_dataset.npz` — padded embedding sequences + lengths + labels + split indices
- `{RUN_TAG}_embeddings.npz` — template embeddings for reuse
- `{RUN_TAG}_dataset_meta.json` — metadata (dims, counts, split sizes)

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── 6a. Save graph dataset (PyG) ─────────────────────────────────────────────
graph_path = PROCESSED_DIR / f"{RUN_TAG}_graph_dataset.pt"
torch.save({
    "data_list":  all_data,
    "idx_train":  idx_train,
    "idx_val":    idx_val,
    "idx_test":   idx_test,
    "node_dim":   NODE_DIM,
    "edge_dim":   EDGE_DIM,
    "embed_dim":  EMBED_DIM,
}, graph_path)
print(f"Graph dataset saved → {graph_path}  ({graph_path.stat().st_size / 1e6:.1f} MB)")

# ── 6b. Save compact sequence dataset (int16 cluster-ids + embedding table) ──
seq_path = PROCESSED_DIR / f"{RUN_TAG}_seq_dataset.npz"
np.savez_compressed(seq_path,
    cids=seq_cids,                                # (N, MAX_SEQ_LEN) int16
    lengths=seq_lengths,                           # (N,)
    y=seq_y,                                       # (N,)
    window_ids=np.array(seq_block_ids, dtype=object),
    idx_train=idx_train, idx_val=idx_val, idx_test=idx_test,
    max_seq_len=MAX_SEQ_LEN, embed_dim=EMBED_DIM,
    pad_id=PAD_ID,
    embedding_table=hybrid_embeddings,             # (num_templates, EMBED_DIM)
    embedding_cids=np.array(all_cids),             # ordered cluster ids
)
print(f"Seq dataset saved  → {seq_path}  ({seq_path.stat().st_size / 1e6:.1f} MB)")

# ── 6c. Save embeddings for reuse ─────────────────────────────────────────────
emb_path = PROCESSED_DIR / f"{RUN_TAG}_embeddings.npz"
np.savez_compressed(emb_path,
    cluster_ids=np.array(all_cids),
    embeddings=hybrid_embeddings,
    tfidf_vocab=json.dumps(dict(tfidf_vec.vocabulary_)),
)
print(f"Embeddings saved   → {emb_path}  ({emb_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# ── 6d. Save metadata ─────────────────────────────────────────────────────────
meta = {
    "run_tag":         RUN_TAG,
    "dataset":         "BGL",
    "labels_source":   "inline (first token of log line)",
    "total_windows":   len(all_data),
    "total_anomalous": int(all_labels.sum()),
    "anomaly_rate":    float(all_labels.mean()),
    "num_templates":   len(templates_data),
    "embed_dim":       EMBED_DIM,
    "node_dim":        NODE_DIM,
    "edge_dim":        EDGE_DIM,
    "max_seq_len":     int(MAX_SEQ_LEN),
    "seed":            SEED,
    "splits": {
        "train": {"size": len(idx_train), "anomalous": int(all_labels[idx_train].sum())},
        "val":   {"size": len(idx_val),   "anomalous": int(all_labels[idx_val].sum())},
        "test":  {"size": len(idx_test),  "anomalous": int(all_labels[idx_test].sum())},
    },
}

meta_path = PROCESSED_DIR / f"{RUN_TAG}_dataset_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Metadata saved     → {meta_path}")
print(json.dumps(meta, indent=2))


In [ ]:
# ── Quick reload test ─────────────────────────────────────────────────────────
print("=== Reload verification ===")

# Graph dataset
loaded = torch.load(graph_path, weights_only=False)
print(f"Graph: {len(loaded['data_list']):,} graphs, "
      f"node_dim={loaded['node_dim']}, edge_dim={loaded['edge_dim']}")
print(f"  train={len(loaded['idx_train']):,}, val={len(loaded['idx_val']):,}, "
      f"test={len(loaded['idx_test']):,}")

# Sequence dataset (compact format)
loaded_seq = np.load(seq_path, allow_pickle=True)
print(f"\nSeq (compact): cids={loaded_seq['cids'].shape}, y={loaded_seq['y'].shape}, "
      f"max_seq_len={loaded_seq['max_seq_len']}")
print(f"  Embedding table: {loaded_seq['embedding_table'].shape}")
print(f"  Disk size: {seq_path.stat().st_size / 1e6:.1f} MB")

# Demonstrate dense reconstruction for one sample
sample_cids = loaded_seq['cids'][0]
emb_table   = loaded_seq['embedding_table']
emb_cid_map = {int(c): i for i, c in enumerate(loaded_seq['embedding_cids'])}
dense_row   = np.zeros((len(sample_cids), emb_table.shape[1]), dtype=np.float32)
for j, c in enumerate(sample_cids):
    if c != int(loaded_seq['pad_id']):
        dense_row[j] = emb_table[emb_cid_map[int(c)]]
print(f"  Dense reconstruction sample: {dense_row.shape}")

# Embeddings
loaded_emb = np.load(emb_path, allow_pickle=True)
print(f"\nEmbeddings: {loaded_emb['embeddings'].shape} for cids {loaded_emb['cluster_ids'][:5]}...")

print("\n✓ All files reload successfully")


---
## Usage in downstream notebooks

### Graph-first GNN (train_gat.ipynb / graph_autoencoder.ipynb)
```python
import torch
from torch_geometric.loader import DataLoader

ds = torch.load("../data/processed/{RUN_TAG}_graph_dataset.pt", weights_only=False)
train_loader = DataLoader([ds['data_list'][i] for i in ds['idx_train']], batch_size=64, shuffle=True)
val_loader   = DataLoader([ds['data_list'][i] for i in ds['idx_val']],   batch_size=128)
test_loader  = DataLoader([ds['data_list'][i] for i in ds['idx_test']],  batch_size=128)

NODE_DIM = ds['node_dim']  # input feature size
EDGE_DIM = ds['edge_dim']  # edge feature size
```

### Sequence-first model (LSTM / Transformer) — compact format
```python
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

ds = np.load("../data/processed/{RUN_TAG}_seq_dataset.npz", allow_pickle=True)

# Build embedding lookup (cluster_id → row index)
emb_table = torch.from_numpy(ds['embedding_table'])   # (num_templates, EMBED_DIM)
cid_to_idx = {int(c): i for i, c in enumerate(ds['embedding_cids'])}
PAD_ID = int(ds['pad_id'])

class SeqDataset(Dataset):
    """Reconstructs dense embeddings on-the-fly from int16 cluster-id sequences."""
    def __init__(self, cids, lengths, labels):
        self.cids    = cids      # (N, MAX_SEQ_LEN) int16
        self.lengths = lengths
        self.labels  = labels

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, idx):
        row = self.cids[idx]
        dense = torch.zeros(len(row), emb_table.shape[1])
        for j, c in enumerate(row):
            if c != PAD_ID:
                dense[j] = emb_table[cid_to_idx[int(c)]]
        return dense, self.lengths[idx], self.labels[idx]

train_ds = SeqDataset(ds['cids'][ds['idx_train']],
                      ds['lengths'][ds['idx_train']],
                      ds['y'][ds['idx_train']])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
```